# SOEN 363 Assignment 2 (Fall 2026): TrailLink Rentals
**Name:** Kaila Quimson  **Student ID:** 40240746

## 0. Imports and setup

In [1]:
import pandas as pd
from IPython.display import HTML

# helper to show a table at its natural size (not stretched)
def show(df):
    if df.empty:
        print("(empty result)")
        return
    html = df.to_html(index=False)
    html = html.replace("<table", '<table style="width:auto"', 1)  # shrink to fit
    display(HTML(html))

## 1. Task 1: Data loading
### 1.1 Load the three files

In [2]:
# equipment comes from a CSV file
equipment_raw = pd.read_csv("Tables/equipment.csv")

# members comes from an Excel file (needs openpyxl)
members_raw = pd.read_excel("Tables/members.xlsx", engine="openpyxl")

# rentals is a .txt file but it's comma-separated, so read_csv works
rentals_raw = pd.read_csv("Tables/rentals.txt", sep=",")

# quick summary to check everything loaded
# each key ("table", "file", ...) becomes a column header; each list fills that column
summary = pd.DataFrame({
    "table": ["equipment", "members", "rentals"],
    "file": ["equipment.csv", "members.xlsx", "rentals.txt"],
    "rows": [len(equipment_raw), len(members_raw), len(rentals_raw)],
    "columns": [equipment_raw.shape[1], members_raw.shape[1], rentals_raw.shape[1]],
})
show(summary)

table,file,rows,columns
equipment,equipment.csv,17,5
members,members.xlsx,14,4
rentals,rentals.txt,67,6


### 1.2 First five rows

In [3]:
print("Equipment")
show(equipment_raw.head())

print("Members")
show(members_raw.head())

print("Rentals")
show(rentals_raw.head())

Equipment


equipment_id,equipment_name,category,daily_rate,stock_quantity
1,City Bike,Cycling,25,12
2,Mountain Bike,Cycling,45,8
3,Road Bike,Cycling,65,5
4,Kayak,Water Sports,55,6
5,Paddleboard,Water Sports,40,7


Members


member_id,member_name,email,city
1,Alex Roy,member1@example.com,Montreal
2,Sam Chen,member2@example.com,Ottawa
3,Maya Patel,member3@example.com,Halifax
4,Noah Martin,member4@example.com,Montreal
5,Lina Ali,member5@example.com,Ottawa


Rentals


rental_id,member_id,equipment_id,quantity,rental_days,rental_date
1,3,1,5.0,2,2026-09-03
2,2,9,6.0,4,2026-08-25
3,7,3,6.0,2,2026-09-07
4,1,2,2.0,5,2026-09-05
5,11,11,2.0,5,2026-10-11


### 1.3 Column names, non-null counts and dtypes

In [4]:
# same info as .info(), but as a table: column, type, non-null and missing counts
def column_info(df):
    return pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "non-null": df.notna().sum().values,
        "missing": df.isna().sum().values,
    })

print("Equipment")
show(column_info(equipment_raw))

print("Members")
show(column_info(members_raw))

print("Rentals")
show(column_info(rentals_raw))

Equipment


column,dtype,non-null,missing
equipment_id,int64,17,0
equipment_name,str,17,0
category,str,16,1
daily_rate,int64,17,0
stock_quantity,int64,17,0


Members


column,dtype,non-null,missing
member_id,int64,14,0
member_name,str,14,0
email,str,13,1
city,str,14,0


Rentals


column,dtype,non-null,missing
rental_id,int64,67,0
member_id,int64,67,0
equipment_id,int64,67,0
quantity,float64,66,1
rental_days,int64,67,0
rental_date,str,67,0


## 2. Task 2: Data cleaning
Cleaning is done on copies of the raw tables. Each step logs how many rows it removed (including 0), and steps run in the required order.
### 2.0 Working copies and audit log

In [5]:
# work on copies so the original data stays untouched
equipment = equipment_raw.copy()
members = members_raw.copy()
rentals = rentals_raw.copy()

# audit log: one row per table per step
audit = []

# record how many rows a step removed
def log(table, step, before, after):
    audit.append({"table": table, "step": step, "before": before,
                  "removed": before - after, "after": after})

# starting row counts
log("equipment", "0. Initial", len(equipment), len(equipment))
log("members", "0. Initial", len(members), len(members))
log("rentals", "0. Initial", len(rentals), len(rentals))

# show the starting counts
show(pd.DataFrame(audit))

table,step,before,removed,after
equipment,0. Initial,17,0,17
members,0. Initial,14,0,14
rentals,0. Initial,67,0,67


### 2.1 Step 1: Remove rows with missing values

In [6]:
# equipment: drop rows with any empty cell
before = len(equipment)
equipment = equipment.dropna()
log("equipment", "1. Missing values", before, len(equipment))

# members: same
before = len(members)
members = members.dropna()
log("members", "1. Missing values", before, len(members))

# rentals: same
before = len(rentals)
rentals = rentals.dropna()
log("rentals", "1. Missing values", before, len(rentals))

# show this step's results: last 3 rows of the audit (step name is in the heading above)
show(pd.DataFrame(audit).tail(3).drop(columns="step"))

table,before,removed,after
equipment,17,1,16
members,14,1,13
rentals,67,1,66


### 2.2 Step 2: Remove exact duplicate rows (keep first)
Duplicates are checked on all columns. Repeat rentals of the same member and equipment are valid and are kept.

In [7]:
# equipment: drop rows that are exact copies, keep the first one
before = len(equipment)
equipment = equipment.drop_duplicates(keep="first")
log("equipment", "2. Exact duplicates", before, len(equipment))

# members: same
before = len(members)
members = members.drop_duplicates(keep="first")
log("members", "2. Exact duplicates", before, len(members))

# rentals: compares all columns, so repeat rentals (same member + equipment) stay
before = len(rentals)
rentals = rentals.drop_duplicates(keep="first")
log("rentals", "2. Exact duplicates", before, len(rentals))

# show this step's results: last 3 rows of the audit (step name is in the heading above)
show(pd.DataFrame(audit).tail(3).drop(columns="step"))

table,before,removed,after
equipment,16,1,15
members,13,1,12
rentals,66,1,65


### 2.3 Step 3: Validate numeric values
Numbers are converted with `errors="coerce"` (bad values become missing). IDs, quantity and rental_days must be positive integers, daily_rate ≥ 1.00, and stock_quantity ≥ 0 (zero stock is valid).

In [8]:
# True when the value exists and is a whole number (5.0 counts as 5)
def is_whole(col):
    return col.notna() & (col == col.round())

# --- equipment ---
before = len(equipment)

# convert to numbers; anything invalid becomes missing
equipment["equipment_id"] = pd.to_numeric(equipment["equipment_id"], errors="coerce")
equipment["daily_rate"] = pd.to_numeric(equipment["daily_rate"], errors="coerce")
equipment["stock_quantity"] = pd.to_numeric(equipment["stock_quantity"], errors="coerce")

# rules: positive whole ID, rate at least 1.00, stock 0 or more
valid = (
    is_whole(equipment["equipment_id"]) & (equipment["equipment_id"] > 0)
    & (equipment["daily_rate"] >= 1.00)
    & is_whole(equipment["stock_quantity"]) & (equipment["stock_quantity"] >= 0)
)
equipment = equipment[valid]   # keep only valid rows

# store whole numbers as int, money as float
equipment = equipment.astype({"equipment_id": int, "daily_rate": float, "stock_quantity": int})
log("equipment", "3. Numeric checks", before, len(equipment))

# --- members ---
before = len(members)

# convert ID to a number
members["member_id"] = pd.to_numeric(members["member_id"], errors="coerce")

# rule: positive whole ID
valid = is_whole(members["member_id"]) & (members["member_id"] > 0)
members = members[valid]   # keep only valid rows
members = members.astype({"member_id": int})
log("members", "3. Numeric checks", before, len(members))

# --- rentals ---
before = len(rentals)

# convert all number columns
rentals["rental_id"] = pd.to_numeric(rentals["rental_id"], errors="coerce")
rentals["member_id"] = pd.to_numeric(rentals["member_id"], errors="coerce")
rentals["equipment_id"] = pd.to_numeric(rentals["equipment_id"], errors="coerce")
rentals["quantity"] = pd.to_numeric(rentals["quantity"], errors="coerce")
rentals["rental_days"] = pd.to_numeric(rentals["rental_days"], errors="coerce")

# rule: every one must be a positive whole number
valid = (
    is_whole(rentals["rental_id"]) & (rentals["rental_id"] > 0)
    & is_whole(rentals["member_id"]) & (rentals["member_id"] > 0)
    & is_whole(rentals["equipment_id"]) & (rentals["equipment_id"] > 0)
    & is_whole(rentals["quantity"]) & (rentals["quantity"] > 0)
    & is_whole(rentals["rental_days"]) & (rentals["rental_days"] > 0)
)
rentals = rentals[valid]   # keep only valid rows

# quantity was float (5.0), now back to int
rentals = rentals.astype({"rental_id": int, "member_id": int, "equipment_id": int,
                          "quantity": int, "rental_days": int})
log("rentals", "3. Numeric checks", before, len(rentals))

# show this step's results: last 3 rows of the audit (step name is in the heading above)
show(pd.DataFrame(audit).tail(3).drop(columns="step"))

table,before,removed,after
equipment,15,2,13
members,12,0,12
rentals,65,2,63


### 2.4 Step 4: Parse rental_date and remove invalid dates
Only rentals have a date column; the other tables are logged with 0 removals.

In [9]:
# no date column in equipment or members, so log 0
log("equipment", "4. Invalid dates (n/a)", len(equipment), len(equipment))
log("members", "4. Invalid dates (n/a)", len(members), len(members))

# rentals: parse YYYY-MM-DD; impossible dates (like Feb 30) become missing
before = len(rentals)
rentals["rental_date"] = pd.to_datetime(rentals["rental_date"], format="%Y-%m-%d", errors="coerce")
rentals = rentals.dropna(subset=["rental_date"])   # drop the bad dates
log("rentals", "4. Invalid dates", before, len(rentals))

# show this step's results: last 3 rows of the audit (step name is in the heading above)
show(pd.DataFrame(audit).tail(3).drop(columns="step"))

table,before,removed,after
equipment,13,0,13
members,12,0,12
rentals,63,1,62


### 2.5 Step 5: Remove rentals with missing parent records

In [10]:
# parent tables have no foreign keys, so log 0
log("equipment", "5. Orphan foreign keys (n/a)", len(equipment), len(equipment))
log("members", "5. Orphan foreign keys (n/a)", len(members), len(members))

# keep a rental only if its member AND equipment exist in the cleaned tables
before = len(rentals)
rentals = rentals[
    rentals["member_id"].isin(members["member_id"])
    & rentals["equipment_id"].isin(equipment["equipment_id"])
]
log("rentals", "5. Orphan foreign keys", before, len(rentals))

# show this step's results: last 3 rows of the audit (step name is in the heading above)
show(pd.DataFrame(audit).tail(3).drop(columns="step"))

table,before,removed,after
equipment,13,0,13
members,12,0,12
rentals,62,2,60


### 2.6 Step 6: Final counts, primary key check and full audit

In [11]:
# full audit table, grouped by table then step
audit_df = pd.DataFrame(audit).sort_values(["table", "step"])
show(audit_df)

# final summary: row counts + primary key check (True = no repeated IDs)
final_check = pd.DataFrame({
    "table": ["equipment", "members", "rentals"],
    "final rows": [len(equipment), len(members), len(rentals)],
    "primary key": ["equipment_id", "member_id", "rental_id"],
    "PK unique": [equipment["equipment_id"]. is_unique, members["member_id"].is_unique, rentals["rental_id"].is_unique],
})
show(final_check)

table,step,before,removed,after
equipment,0. Initial,17,0,17
equipment,1. Missing values,17,1,16
equipment,2. Exact duplicates,16,1,15
equipment,3. Numeric checks,15,2,13
equipment,4. Invalid dates (n/a),13,0,13
equipment,5. Orphan foreign keys (n/a),13,0,13
members,0. Initial,14,0,14
members,1. Missing values,14,1,13
members,2. Exact duplicates,13,1,12
members,3. Numeric checks,12,0,12


table,final rows,primary key,PK unique
equipment,13,equipment_id,True
members,12,member_id,True
rentals,60,rental_id,True


**Effect on joins:** Each primary key is now unique, so every rental matches exactly one member and one equipment row, and nothing gets duplicated when joining. Rentals pointing to members or equipment that don't exist were removed, so inner joins won't silently drop rows, and invalid numbers or dates can't distort revenue calculations.